# fig4-strf-performance-low: encoding models on Wehr's A1 recordings (CRCNS ac-1)

Paper: Asari and Zador (2009), J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009. Data set: CRCNS ac-1 (doi:10.6080/K0KW5CXR), unversioned, read from the public bucket `crcnsarchive` (`ac-1/crcns-ac1.zip`, MD5 2300ee76d93383ab8c808ddf16761a23 in the data set's `checksums.md5`), the `wehr/` folder. CRCNS does not version data sets, so the version is recorded as `unversioned` with the MD5 sums of the files read. The claims file was written from the version of record. This notebook tests one claim; the other claims are in `reproduce_asari_2009.ipynb`.

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of the predictable response power, and extending the window from 180 ms to about 4 s did not improve them significantly, with or without a static nonlinearity; the LN model did only slightly better.

In [ ]:
import time, zlib, resource
import numpy as np
import pandas as pd
from scipy import stats
import wehr as W
pd.set_option("display.width", 200)
cells = W.cells()

## Data check

In [ ]:
rows = []
for c in cells:
    rep = {k: len(v) for k, v in c["stimuli"].items()}
    rows.append(dict(cell=c["cell"], experiment=c["experiment"][:45], stimuli=len(rep), stimuli_2plus_trials=sum(v >= 2 for v in rep.values()),
                     trials_min=min(rep.values()) if rep else 0, trials_max=max(rep.values()) if rep else 0))
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
use = [c for c in cells if any(len(v) >= 2 for v in c["stimuli"].values())]
print("cells in wehr/Summary.mat:", len(cells), "| with at least two trials of a sound (Eq. 52 needs repeats):", len(use))
tr, sf, t0, dur, sfile = W.load_trial(next(iter(use[0]["stimuli"].values()))[0])
print("example trial: %d samples at %.0f Hz (%.1f s), sound onset at sample %d, duration %.0f ms, stimulus %s; range %.1f to %.1f mV" % (len(tr), sf, len(tr) / sf, t0, dur, sfile, tr.min(), tr.max()))

The traces are membrane potential in mV at 4 kHz (`response.sf`), one file per trial, with the sound onset in `triggers.time`; the cells of the second set (2003) have at most two trials of each sound and the first set (2002) 6 to 25. The paper names 20 cells of Machens et al. (2004) without saying which; the 16 cells with at least two trials of a sound are used, since Eq. 52 needs repeats.

Clock check: the model's validation performance with the true stimulus, against the same responses paired with the spectrogram shifted by 2 s in time (circularly within each sound), for the shortest window, shows that the responses follow the stimulus with the trigger times as given (printed in the claim cell).

In [ ]:
t_start = time.time()
res = []
variants = []
clock = []
for c in use:
    stims = []
    for stem, paths in c["stimuli"].items():
        if len(paths) < 2:
            continue
        R, sfile = W.responses(paths)
        x, sfs = W.load_stimulus(sfile)
        stims.append((R, W.spectrogram(x, sfs, 10.0), x, sfs))
    main = W.fit_cell([(R, S) for R, S, x, s_ in stims])
    for r in main:
        res.append(dict(cell=c["cell"], **r))
    # readings at the shortest and longest windows: 20-ms Hann window; folds by whole sound
    two = [W.CONFIGS[0], W.CONFIGS[-1]]
    S20 = [(R, W.spectrogram(x, s_, 20.0)) for R, S, x, s_ in stims]
    for name, st, folds in [("hann 20 ms", S20, "contiguous"), ("folds by sound", [(R, S) for R, S, x, s_ in stims], "stimulus")]:
        for r in W.fit_cell(st, folds=folds, configs=two):
            variants.append(dict(cell=c["cell"], reading=name, **r))
    sh = [(R, np.roll(S, int(2 / W.DT), axis=1)) for R, S, x, s_ in stims]
    clock.append(dict(cell=c["cell"], true=main[0]["strf_valid"], shifted=W.fit_cell(sh, configs=[W.CONFIGS[0]], with_ln=False)[0]["strf_valid"]))
    del stims, S20, sh
    print(c["cell"], "done, %.0f s" % (time.time() - t_start), flush=True)
res = pd.DataFrame(res); variants = pd.DataFrame(variants); clock = pd.DataFrame(clock)
print("peak memory %.1f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
print("clock check (validation performance, 180-ms window): true %.3f, stimulus shifted 2 s %.3f (mean over %d cells)" % (clock.true.mean(), clock.shifted.mean(), len(clock)))

In [ ]:
summ = res.groupby("window_ms")[["strf_valid", "strf_train", "ln_valid", "ln_train"]].agg(["mean", "std"])
print(summ.round(3).to_string())
print("ridge parameter interior to the grid in", int(res.ridge_interior.sum()), "of", len(res), "fits")
mx = res.groupby("window_ms")[["strf_valid", "strf_train", "ln_valid", "ln_train"]].mean()
print("largest mean over windows: STRF validation %.3f, STRF training %.3f, LN validation %.3f, LN training %.3f" % tuple(mx.max()))
print("mean LN minus STRF (validation) over windows and cells: %.3f" % (res.ln_valid - res.strf_valid).mean())
a = res[res.window_ms == 180].set_index("cell"); b = res[res.window_ms == 4052].set_index("cell")
for model in ["strf", "ln"]:
    d = b[f"{model}_valid"] - a[f"{model}_valid"]
    w = stats.wilcoxon(b[f"{model}_valid"], a[f"{model}_valid"], alternative="greater")
    print(f"{model.upper()}: validation 4052 vs 180 ms, median change {d.median():.3f} (mean {d.mean():.3f}), {int((d > 0).sum())} of {len(d)} cells improve; one-sided Wilcoxon p = {w.pvalue:.4g}")
    piv = res.pivot(index="cell", columns="window_ms", values=f"{model}_valid")
    fr = stats.friedmanchisquare(*[piv[c] for c in piv.columns])
    print(f"   Friedman over the nine windows (variant): chi2 = {fr.statistic:.2f}, p = {fr.pvalue:.4g}")
print("by cell set:")
res["set"] = np.where(res.cell.str.startswith("2002"), "2002 (repeats)", "2003 (two trials)")
print(res.groupby(["set", "window_ms"])[["strf_valid", "ln_valid"]].mean().round(3).to_string())
print("readings at 180 and 4052 ms:")
print(variants.groupby(["reading", "window_ms"])[["strf_valid", "strf_train", "ln_valid", "ln_train"]].mean().round(3).to_string())
for name, g in variants.groupby("reading"):
    a2 = g[g.window_ms == 180].set_index("cell"); b2 = g[g.window_ms == 4052].set_index("cell")
    for model in ["strf", "ln"]:
        w = stats.wilcoxon(b2[f"{model}_valid"], a2[f"{model}_valid"], alternative="greater")
        print(f"  {name}, {model.upper()}: median change {(b2[f'{model}_valid'] - a2[f'{model}_valid']).median():.3f}, one-sided Wilcoxon p = {w.pvalue:.4g}")

**Result: not reproduced, resting on assumptions.** On the 16 cells with repeats, mean validation performance is 0.106 to 0.143 for the STRF and 0.002 to 0.099 for the LN model at every window (training 0.22 to 0.30 and 0.24 to 0.33), so the validation estimate stays below 0.22 while the training estimate does not. But extending the window from 180 to 4052 ms improves the STRF: 13 of 16 cells improve, median change 0.038, one-sided Wilcoxon p = 0.0031 (Friedman over nine windows p = 0.0044); for the LN model p = 0.058. The claim of absence fails on the STRF under the test the claims file fixed, and under both other readings (p = 0.0038 with a 20-ms window, 0.0145 with folds by sound). The LN model does worse than the STRF on validation data (mean difference -0.068), not slightly better. The cells (16 with repeats, the paper's 20 unnamed) and the test are assumptions; with the 2002 cells alone the STRF changes from 0.098 to 0.124.